# Lab 1 古典 IR：親手做出一個最小搜尋引擎（完整版）

**今天的目標：** 親手做出一個會「打分數、排名次」的最小搜尋引擎——輸入查詢，回傳排序後最相關的文件與分數。它就是下午 RAG「檢索」那一半的原型。

**這本是完整版：** 所有 TODO 已填入答案、關鍵行附 `💡` 註解；小作業附參考解。供回家複習對照——課堂上請先用教學版自己想。

> 語料為**虛構、教學用**財經新聞（非真實行情）。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過；裝不起來看 README）
!pip install -q scikit-learn numpy jieba rank_bm25

## 🔧 第 0 步：環境檢查

先跑下一格，全綠再往下。

In [1]:
# ✅ 環境檢查：跑這格，全部通過再往下（練 A–C 只需要 sklearn / numpy）
import sys
print("Python：", sys.version.split()[0])

import sklearn, numpy
print("scikit-learn：", sklearn.__version__)
print("numpy：", numpy.__version__)

import glob
news_files = sorted(glob.glob("data/news/*.txt"))
n_files = len(news_files)
print(f"data/news/ 語料：找到 {n_files} 篇（預期 15 篇，練 D 才用到）")

if n_files == 15:
    print("")
    print("✅ 環境 OK，可以開始！（jieba 到練 D 開頭才檢查，現在不用管）")
else:
    print("")
    print("❌ 沒找到 15 篇語料——請確認你是在 Lab1_古典IR/ 資料夾底下開這本 notebook，")
    print("   關掉 Jupyter、切到正確資料夾再重開一次。")

Python： 3.12.13


scikit-learn： 1.8.0
numpy： 2.2.6
data/news/ 語料：找到 15 篇（預期 15 篇，練 D 才用到）

✅ 環境 OK，可以開始！（jieba 到練 D 開頭才檢查，現在不用管）


### 🛟 測試套件

這 5 行能跑、印出像 `[0.7xx 0.]`，代表環境沒問題。

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs = ["升息 台股", "原油 庫存"]
v = TfidfVectorizer(token_pattern=r"(?u)\S+")
M = v.fit_transform(docs)
print(cosine_similarity(v.transform(["升息"]), M)[0])   # 應印 [非0, 0]

[0.70710678 0.        ]


---
## 🟦 練 A：手刻倒排索引 + 布林查詢

用 `dict` 刻出**倒排索引**（詞 → 出現在哪些文件），再做布林 AND / OR 查詢。

> ⚠️ 「倒排」的「排」指**方向**倒過來（詞→文件），**跟排名次無關**——排序是練 B 的事。

### A1・準備語料（直接跑，不用改）

5 篇虛構財經新聞，已用空白斷好詞。

In [3]:
# 5 篇【虛構・教學用】財經新聞短句（非真實行情，僅供教學）
# 用空白把詞切開，方便最小版直接用 split() 斷詞（中文真實斷詞見練 D）
docs = [
    "升息 抑制 通膨 央行 升息 一碼",          # 文件0：升息
    "台股 開高 半導體 領漲 台股 收紅",        # 文件1：台股/半導體
    "原油 庫存 下滑 油價 走高",               # 文件2：原油
    "半導體 毛利率 提升 晶圓代工 報價 上揚",   # 文件3：半導體財報
    "升息 衝擊 台股 資金 外流",               # 文件4：升息+台股
]

for i, d in enumerate(docs):   # 💡 enumerate＝一邊跑清單、一邊給編號：(0, 第0篇)、(1, 第1篇)…
    print(f"文件{i}: {d}")     #    倒排索引要記「哪一篇」，所以非得有編號不可

文件0: 升息 抑制 通膨 央行 升息 一碼
文件1: 台股 開高 半導體 領漲 台股 收紅
文件2: 原油 庫存 下滑 油價 走高
文件3: 半導體 毛利率 提升 晶圓代工 報價 上揚
文件4: 升息 衝擊 台股 資金 外流


### A2・手刻倒排索引

建出 `{詞: [文件編號]}`。

**預期輸出：** `升息 → [0, 4]`　`台股 → [1, 4]`　`半導體 → [1, 3]`　`原油 → [2]`

### A2-1・先看一眼：一句話怎麼變成「不重複的詞」

建索引之前，先把**單一句話**拆開看清楚。

**預期輸出：** 原文一行，然後是一組**沒有重複**的詞（注意：文件 0 的「升息」出現兩次，這裡只會留一個）。

In [4]:
text = docs[0]
print("原文：", text)

set(text.split())  # 💡 split()＝用空白切成詞；set()＝去重
                   # 💡 為什麼要去重？倒排索引只在乎「這篇有沒有這個詞」，出現幾次不是這一步的事
                   # 最後一行不寫 print，Jupyter 會直接顯示

原文： 升息 抑制 通膨 央行 升息 一碼


{'一碼', '升息', '央行', '抑制', '通膨'}

In [5]:
inverted = {}                      # 倒排索引：{詞: 含此詞的文件編號清單}
for doc_id, text in enumerate(docs):
    for word in set(text.split()): # 上一格剛看過：切詞 + 去重
        if word not in inverted:   # 這個詞第一次出現 → 先給它一個空清單
            inverted[word] = []
        inverted[word].append(doc_id)
        # 💡 「沒看過就先開空清單、再 append」——用 dict 收集清單的標準三步驟

for w in ["升息", "台股", "半導體", "原油"]:
    print(w, "→", sorted(inverted[w]))

升息 → [0, 4]
台股 → [1, 4]
半導體 → [1, 3]
原油 → [2]


### A3・用倒排索引做布林查詢（兩格：先 AND、再 OR）

`AND` ＝各詞 postings 取**交集**；`OR` ＝取**聯集**。完全不用掃原文。

**預期輸出：** `升息 AND 台股 → [4]`　／　`升息 OR 台股 → [0, 1, 4]`

> 跑完想一下：它只回「哪些符合」，**沒告訴你哪篇比較相關**——這就是布林的極限。

### A3-0・先練一次「集合三兄弟」：交集 / 聯集 / 差集

布林查詢**整個都是集合運算**——所以先把三個運算單獨玩過一次，等一下才不會卡在語法上。

直接拿剛剛建好的 postings 來玩：**「升息」在哪幾篇**、**「台股」在哪幾篇**。

**預期輸出：**
```
升息 → {0, 4}
台股 → {1, 4}

交集（兩邊都有）    ： {4}          ← 只有文件 4 同時談升息和台股
聯集（任一邊有）    ： {0, 1, 4}
差集（升息有、台股沒）： {0}
```

In [6]:
# 集合三兄弟：交集 / 聯集 / 差集——布林查詢的骨架
a = set(inverted["升息"])   # 哪幾篇有「升息」
b = set(inverted["台股"])   # 哪幾篇有「台股」
print("升息 →", a)
print("台股 →", b)
print()

print("交集（兩邊都有）    ：", a & b)   # 💡 & ＝交集 → 這就是 AND
print("聯集（任一邊有）    ：", a | b)   # 💡 | ＝聯集 → 這就是 OR
print("差集（升息有、台股沒）：", a - b)  # 💡 - ＝相減 → NOT 會用到
# 💡 三個運算子都只吃「集合」——postings 本體是 list，用之前一定要先 set() 一下

升息 → {0, 4}
台股 → {1, 4}

交集（兩邊都有）    ： {4}
聯集（任一邊有）    ： {0, 1, 4}
差集（升息有、台股沒）： {0}


In [7]:
# A3-1：boolean_and——AND＝取各詞 postings 的「交集」
def boolean_and(word_list):
    if len(word_list) == 0:
        return []
    result = set(inverted.get(word_list[0], []))   # 第一個詞的 postings，轉成集合當起點
    for w in word_list:
        postings = set(inverted.get(w, []))        # 💡 先轉成集合再運算——postings 本體是 list，list 不能做 & / |
        result = result & postings                 # 💡 & ＝交集：兩邊都有的才留下
    # get(w, [])：查「不存在的詞」回空清單而不是報錯——空集合一交集，結果正確變空
    return sorted(result)

print("升息 →        ", inverted.get("升息", []))
print("台股 →        ", inverted.get("台股", []))
print("升息 AND 台股 →", boolean_and(["升息", "台股"]))

升息 →         [0, 4]
台股 →         [1, 4]
升息 AND 台股 → [4]


In [8]:
# A3-2：boolean_or——OR＝取各詞 postings 的「聯集」
def boolean_or(word_list):
    result = set()                                 # 從空集合開始收
    for w in word_list:
        postings = set(inverted.get(w, []))        # 💡 同 A3-1：先轉集合再運算
        result = result | postings                 # 💡 | ＝聯集：任一邊有就收（集合自動去重，同一篇不會收兩次）
    return sorted(result)

print("升息 OR 台股 → ", boolean_or(["升息", "台股"]))

升息 OR 台股 →  [0, 1, 4]


### 📝 小作業 A

1. **先在紙上預測、再跑 code 驗證：** 「半導體 AND 台股」會回哪幾篇？「原油 OR 半導體」呢？（對照 A2 印出的 postings 表用眼睛算；記得參數是「詞的清單」）
2. **⭐ 進階（選做）：** 寫一個 `boolean_not(word)`，回傳「**不含**該詞」的文件編號。想一想：AND／OR 只要有 postings 就夠，**NOT 需要一個 AND／OR 都不需要的東西**——那是什麼？（提示：AND／OR 只翻 postings 就夠；NOT 得先知道「總共有哪些文件」。集合三兄弟的第三個 `-`＝相減，「全部文件的編號」可以從 `docs` 的長度生出來。）

<details><summary>📖 做完再看：參考解（參考解不只一種，思路對就好）</summary>

```python
# Q1：先用 postings 表眼睛算——
#   半導體 → [1, 3]、台股 → [1, 4] → AND（交集）應該只有文件 1
#   原油 → [2]、半導體 → [1, 3]   → OR（聯集）應該是 [1, 2, 3]
print("半導體 AND 台股 →", boolean_and(["半導體", "台股"]))
print("原油 OR 半導體  →", boolean_or(["原油", "半導體"]))

# ⭐ 進階：NOT ＝ 從「全部文件」扣掉含此詞的（全集 - postings）
def boolean_not(word):
    all_ids = set(range(len(docs)))            # 全部文件的編號 {0,1,2,3,4}——NOT 需要「全集」當基準，這正是 AND/OR 都不需要的東西
    has_word = set(inverted.get(word, []))     # 含這個詞的文件（轉成集合）
    return sorted(all_ids - has_word)          # - ＝集合相減：從全集扣掉

print("NOT 升息       →", boolean_not("升息"))   # 不含「升息」的應為 [1, 2, 3]
```
輸出：`半導體 AND 台股 → [1]`／`原油 OR 半導體 → [1, 2, 3]`／`NOT 升息 → [1, 2, 3]`
</details>

In [9]:
# 📝 小作業 A 參考解（參考解不只一種）

# Q1：先在紙上用 postings 表眼睛算，再跑 code 驗證——
#   半導體 → [1, 3]、台股 → [1, 4]  → AND（交集）應只有文件 1
#   原油 → [2]、半導體 → [1, 3]     → OR（聯集）應是 [1, 2, 3]
print("半導體 AND 台股 →", boolean_and(["半導體", "台股"]))
print("原油 OR 半導體  →", boolean_or(["原油", "半導體"]))

# ⭐ 進階：NOT ＝ 從「全部文件」扣掉含此詞的（全集 - postings）
def boolean_not(word):
    all_ids = set(range(len(docs)))          # 💡 全部文件的編號 {0,1,2,3,4}——NOT 需要「全集」當基準，這正是 AND/OR 都不需要的東西
    has_word = set(inverted.get(word, []))   # 💡 含這個詞的文件（postings 是 list，先轉 set 才能做集合相減）
    return sorted(all_ids - has_word)        # 💡 - ＝集合相減：從全集扣掉

print("NOT 升息       →", boolean_not("升息"))   # 不含「升息」的應為 [1, 2, 3]

# 結論：AND / OR 只要翻 postings 就夠；NOT 多需要一個「全集」當基準（用 docs 的長度生出來）。

半導體 AND 台股 → [1]
原油 OR 半導體  → [1, 2, 3]
NOT 升息       → [1, 2, 3]


---
## 🟩 練 B：TF-IDF 向量化 + cosine 排序（本 Lab 心臟）

把文件變成 **TF-IDF 向量**，查詢也變向量，用 **cosine** 算相似度 → 排序。做完你就有一個會「打分數、排名次」的最小搜尋引擎。

> ⚠️ **sklearn 的分數跟手算對不上是正常的**（它用平滑版 IDF ＋ 正規化）。**不要對數字，看排序。**

### B1・`TfidfVectorizer`：把文件變向量

**預期輸出：** 詞彙表 23、向量矩陣 `(5, 23)`

> 🛟 跑出 `(5, 0)` → 漏改 `token_pattern`，中文被吃掉了。

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# token_pattern：告訴 sklearn「怎麼把一句話切成詞」。
#   預設規則只抓「長度≥2 的英數詞」→ 中文整段被吃掉、shape 變 (5, 0)、一個詞都沒抓到。
#   r"(?u)\S+" ＝「用空白切，中文也算數」——正好對上我們已用空白切好的語料。
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\S+")

doc_vectors = vectorizer.fit_transform(docs)
# 💡 fit＝學詞彙表＋算 IDF；transform＝把每篇轉成向量。fit_transform ＝兩件事一次做完。
# 💡 全 Lab 最重要的規則：**文件用 fit_transform（要「學」）、查詢用 transform（照學過的轉換）**
#    ——練 D 換語料時還會再考一次。

print("詞彙表大小：", len(vectorizer.get_feature_names_out()))
print("向量矩陣形狀：", doc_vectors.shape, "（5 篇 × 詞彙數）")
print("前 8 個詞：", vectorizer.get_feature_names_out()[:8])

詞彙表大小： 23
向量矩陣形狀： (5, 23) （5 篇 × 詞彙數）
前 8 個詞： ['一碼' '上揚' '下滑' '升息' '半導體' '原油' '台股' '報價']


### B2・印 IDF：親眼看「罕見詞分數高」（兩格）

**預期輸出：** 「毛利率」「原油」（各只出現 1 篇）IDF ＝ **2.099**；「升息」「台股」「半導體」（各 2 篇）＝ **1.693**。

> **含詞文件越少 → IDF 越大 → 該詞越有鑑別力。**

In [11]:
# B2-1：先直接看一眼 IDF 的原始長相
vocab = vectorizer.get_feature_names_out()   # 💡 fit 之後才有——這就是 B1 印「前 8 個詞」用的那個方法

print("詞彙表（23 個詞）：")
print(vocab)
print()
print("每個詞的 IDF（23 個數字，跟上面的詞彙表順序一一對應）：")

詞彙表（23 個詞）：
['一碼' '上揚' '下滑' '升息' '半導體' '原油' '台股' '報價' '外流' '央行' '庫存' '抑制' '提升' '收紅'
 '晶圓代工' '毛利率' '油價' '衝擊' '資金' '走高' '通膨' '開高' '領漲']

每個詞的 IDF（23 個數字，跟上面的詞彙表順序一一對應）：


In [12]:
vectorizer.idf_
# 💡 尾巴帶底線的屬性是 sklearn 慣例：「fit 之後才學出來」的結果（沒 fit 過就存取會報錯）
# 最後一行不寫 print——Jupyter 會直接顯示

array([2.09861229, 2.09861229, 2.09861229, 1.69314718, 1.69314718,
       2.09861229, 1.69314718, 2.09861229, 2.09861229, 2.09861229,
       2.09861229, 2.09861229, 2.09861229, 2.09861229, 2.09861229,
       2.09861229, 2.09861229, 2.09861229, 2.09861229, 2.09861229,
       2.09861229, 2.09861229, 2.09861229])

In [13]:
# B2-2：挑幾個詞，把 IDF 印出來對照（這格不用填——直接跑，確認方向）
idf = vectorizer.idf_

for w in ["升息", "台股", "半導體", "毛利率", "原油"]:
    j = list(vocab).index(w)                # 這個詞在詞彙表的第幾格
    value = round(idf[j], 3)                # round(x, 3)：四捨五入到小數第 3 位
    print(w, "IDF =", value)

升息 IDF = 1.693
台股 IDF = 1.693
半導體 IDF = 1.693
毛利率 IDF = 2.099
原油 IDF = 2.099


### B3・查詢向量化 + cosine 排序（核心一刀）

三格：查詢變向量 → 算 cosine → 排名次。

> ⚠️ **全 Lab 最重要的規則：文件用「重新學」的方法、查詢用「照學過的轉換」的方法。** 練 D 還會考一次。

**預期輸出：** 查詢向量 `(1, 23)`；5 篇分數 `[0.628, 0, 0, 0, 0.389]`；文件 0 第 1、文件 4 第 2。

In [14]:
# B3-1：把查詢也變成向量（必須跟文件在同一個向量空間）
query = "升息"
query_vector = vectorizer.transform([query])   # 💡 用 transform（照文件學過的詞彙表轉換），不是 fit_transform！
#        💡 查詢若重學一份詞彙表，就跟文件不在同一個空間了——格子對不上，cosine 根本沒意義
#        💡 括號坑：transform 吃的是「一批文件」，單一查詢字串也要包成 list（[query]）——
#           直接餵字串會噴 ValueError: Iterable over raw text documents expected...

print("查詢向量形狀：", query_vector.shape, "（1 條查詢 × 詞彙數——跟文件向量同一空間，才能比相似）")
print()
print("攤開來看（23 個數字，跟 B2-1 的詞彙表一一對應）：")

query_vector.toarray()   # .toarray()＝把稀疏矩陣攤成看得見的數字；最後一行不寫 print，Jupyter 會直接顯示

查詢向量形狀： (1, 23) （1 條查詢 × 詞彙數——跟文件向量同一空間，才能比相似）

攤開來看（23 個數字，跟 B2-1 的詞彙表一一對應）：


array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.]])

### B3-1b・試試看多個字：一次轉三條查詢

上一格說「它吃的是**一批**」——那就真的一次餵三條進去看看。

**預期輸出：** 形狀是 `(3, 23)`（3 條查詢 × 23 個詞）。
攤開來看第三列（查「黃金」）——**23 個格子全是 0**，因為語料裡根本沒有「黃金」這個詞。

> 💡 記住這個「全 0」的畫面：等一下 `search()` 要處理的「查無相關文件」，就是它造成的。

In [15]:
query1 = "升息"           # 一個詞
query2 = "升息 台股"      # 兩個詞
query3 = "黃金"           # 語料裡根本沒有的詞

query_vectors = vectorizer.transform([query1, query2, query3])
# 💡 transform 吃的是「一批」——所以 list 裡放幾條，就轉幾條（這也是為什麼單一查詢也得包成 list）

print("形狀：", query_vectors.shape, "（3 條查詢 × 詞彙數）")
print()
print("攤開來看（第 3 列是「黃金」——盯著看它有幾個非 0）：")

query_vectors.toarray()
# 💡 第 3 列 23 個格子全是 0：語料裡沒有「黃金」→ 查詢向量是空的 → 跟任何文件的 cosine 都會是 0
#    這就是 search() 需要「查無相關文件」防呆的原因

形狀： (3, 23) （3 條查詢 × 詞彙數）

攤開來看（第 3 列是「黃金」——盯著看它有幾個非 0）：


array([[0.        , 0.        , 0.        , 1.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.70710678, 0.        ,
        0.        , 0.70710678, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ]])

In [16]:
# B3-2：算 cosine 相似度（查詢 vs 每一篇文件）
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(query_vector, doc_vectors)[0]   # 💡 查詢 vs 每一篇的相似度
#        💡 cosine_similarity 回傳 1×5 矩陣（1 條查詢 × 5 篇）；取 [0] 才是「查詢對每一篇」的那一排分數

scores   # 最後一行不寫 print——Jupyter 會直接顯示這 5 個分數

array([0.62791376, 0.        , 0.        , 0.        , 0.38898761])

In [17]:
# B3-3：把分數排成名次（不用填，直接跑）
# 先組「(分數, 文件編號)」的 tuple 清單，再 sorted 由大到小
score_list = []
for doc_id, score in enumerate(scores):
    score_list.append((score, doc_id))
ranking = sorted(score_list, reverse=True)     # tuple 排序預設比第一個元素（分數）；reverse=True＝由大到小

print(f"查詢：「{query}」的搜尋結果（依相關度排序）")
print("")
rank = 1
for score, doc_id in ranking:
    if score > 0:
        s = round(score, 3)
        text = docs[doc_id]
        print(f"第{rank}名  分數={s}  文件{doc_id}: {text}")
        rank = rank + 1

查詢：「升息」的搜尋結果（依相關度排序）

第1名  分數=0.628  文件0: 升息 抑制 通膨 央行 升息 一碼
第2名  分數=0.389  文件4: 升息 衝擊 台股 資金 外流


### B4・包成 `search()` 函式（不用填，直接跑）

**跑完看：** 查「半導體 毛利率」→ 文件 3 第 1、文件 1 第 2；查「黃金」→ 正確回「查無」。

**到這裡，你已經做出一個會打分數、排名次的最小搜尋引擎了。**

In [18]:
def search(query, top_k=3):
    # 💡 跟 B3 完全同一套流程，只是包成函式＋top_k 截斷＋「查無」防呆——之後重複查、換語料都照用
    qv = vectorizer.transform([query])
    scores = cosine_similarity(qv, doc_vectors)[0]
    score_list = []
    for doc_id, score in enumerate(scores):
        score_list.append((score, doc_id))
    ranking = sorted(score_list, reverse=True)

    print("")
    print(f"查詢：「{query}」")
    shown = 0                                  # 已經印出幾名
    for score, doc_id in ranking:
        if score > 0 and shown < top_k:
            shown = shown + 1
            s = round(score, 3)
            text = docs[doc_id]
            print(f"  第{shown}名 分數={s} 文件{doc_id}: {text}")
    if shown == 0:
        print("  （查無相關文件）")

### B4-2・實際用用看

函式寫好了，就來查幾次。

**預期輸出：** 三次查詢的排名。**盯著第三次**——查「黃金」時，走的是「查無相關文件」那條路（就是 B3-1b 那個全 0 向量造成的）。

In [19]:
search("台股")
search("半導體 毛利率")     # 多詞查詢也行
search("黃金")              # 故意查語料裡沒有的詞 → 走「查無相關文件」那條路


查詢：「台股」
  第1名 分數=0.645 文件1: 台股 開高 半導體 領漲 台股 收紅
  第2名 分數=0.389 文件4: 升息 衝擊 台股 資金 外流

查詢：「半導體 毛利率」
  第1名 分數=0.541 文件3: 半導體 毛利率 提升 晶圓代工 報價 上揚
  第2名 分數=0.203 文件1: 台股 開高 半導體 領漲 台股 收紅

查詢：「黃金」
  （查無相關文件）


### 📝 小作業 B

1. 用 `search()` 查「**原油 庫存**」——先猜第 1 名是哪篇，再跑。
2. 查「**晶圓代工**」——哪幾篇會出現？為什麼只有它？（想想 B2 的 IDF）
3. 把 `top_k` 調成 5，查「**升息 台股**」——**文件 4**（「升息」「台股」各出現 1 次，兩個查詢詞都沾到）和 **文件 0**（只有「升息」，但出現 2 次）誰排前面？先猜再跑，然後想：**「兩個查詢詞都命中」和「一個詞重複兩次」，哪個比較能代表這篇跟查詢有關？**

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
search("原油 庫存")      # Q1：文件 2 同時含「原油」「庫存」→ 穩排第 1
search("晶圓代工")       # Q2：只有文件 3 含此詞——罕見詞 IDF 高，一查就中
search("升息 台股", top_k=5)   # Q3
```

- **Q1：** 文件 2 兩個詞都含 → 第 1 名。
- **Q2：** 「晶圓代工」只出現在文件 3（IDF 最高的那群），所以只有它被撈出來。
- **Q3：** **文件 4（0.55）排在文件 0（0.444）前面**——`升息 台股` 兩個查詢詞文件 4 都沾到，文件 0 只沾到「升息」（就算出現 2 次）。**結論：多詞覆蓋 > 單詞重複。**
  - 🔍 眼尖的話會發現文件 1（0.456）也擠在中間、甚至略高於文件 0——這兩個分數非常接近（0.444 vs 0.456），誰前誰後取決於**向量長度正規化**的細微差異，**超出今天的範圍、也不影響結論**。今天只要抓住「**多詞覆蓋 > 單詞重複**」這一句就好。
</details>

In [20]:
# 📝 小作業 B 參考解（參考解不只一種）

# Q1：文件 2 同時含「原油」「庫存」兩詞 → 穩排第 1
search("原油 庫存")

# Q2：只有文件 3 含「晶圓代工」——罕見詞（IDF 高）只在一篇出現，一查就中它
search("晶圓代工")

# Q3：文件 4（0.55）排在文件 0（0.444）前面——「升息」「台股」兩個查詢詞文件 4 都沾到，
# 文件 0 只沾到「升息」（就算出現 2 次）→ 結論：多詞覆蓋 > 單詞重複。
# （文件 1 的 0.456 和文件 0 的 0.444 非常接近，誰前誰後取決於向量長度正規化的細微差異，
#   超出今天範圍、也不影響結論——今天只要抓住「多詞覆蓋 > 單詞重複」。）
search("升息 台股", top_k=5)


查詢：「原油 庫存」
  第1名 分數=0.632 文件2: 原油 庫存 下滑 油價 走高

查詢：「晶圓代工」
  第1名 分數=0.421 文件3: 半導體 毛利率 提升 晶圓代工 報價 上揚

查詢：「升息 台股」
  第1名 分數=0.55 文件4: 升息 衝擊 台股 資金 外流
  第2名 分數=0.456 文件1: 台股 開高 半導體 領漲 台股 收紅
  第3名 分數=0.444 文件0: 升息 抑制 通膨 央行 升息 一碼


---
## 🟧 練 C：檢索評估 precision / recall

**漁網類比：** 海裡的魚＝真正相關的文件，你撒網撈起一堆（混著魚和垃圾，海裡還有漏網之魚）。
- **precision** ＝ 撈起來的那堆裡，**魚佔多少** → 我撈的**準不準**？
- **recall** ＝ 海裡所有的魚，**被我撈到幾條** → 該撈的**有沒有漏**？

### C1・親手算一次 P / R（兩格：先 precision、再 recall + F1）

**預期輸出：** `Precision = 2/5 = 0.4`　`Recall = 2/4 = 0.5`　`F1 = 0.44`

In [21]:
# C1-1：precision——「我撈的準不準？」（分母＝我撈起來的全部）
# 假設一次查詢：系統回了 5 篇（retrieved），我們事先標好每篇是否「真的相關」
# 全語料中「真正相關」的總共有 4 篇（含沒被撈到的）
retrieved   = ["d1", "d2", "d3", "d4", "d5"]          # 系統撈回的（依分數排序）
relevant    = {"d1", "d3", "d6", "d7"}                # 標準答案：真正相關的全集（共 4 篇）

hit = []                                   # 撈到的當中，真的相關的（✅魚）
for d in retrieved:
    if d in relevant:
        hit.append(d)

precision = len(hit) / len(retrieved)      # 💡 分母＝我撈的全部（撈得多，分母就大）

n_hit = len(hit)
n_retrieved = len(retrieved)
p = round(precision, 2)
print(f"撈到且相關（✅魚）：{hit}")
print(f"Precision = {n_hit}/{n_retrieved} = {p}")

撈到且相關（✅魚）：['d1', 'd3']
Precision = 2/5 = 0.4


In [22]:
# C1-2：recall——「該撈的有沒有漏？」（分母＝海裡所有的魚）＋ F1
recall = len(hit) / len(relevant)          # 💡 分母＝海裡所有的魚（標準答案總數）
#        💡 跟 precision 比一比：**分子一模一樣，只有分母換了**——這就是兩者唯一的差別

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0
# 💡 F1＝precision 與 recall 的調和平均：任一邊掉到 0，F1 就是 0——逼你兩個都顧

n_relevant = len(relevant)
r = round(recall, 2)
f1_value = round(f1, 2)
print(f"Recall = {n_hit}/{n_relevant} = {r}")
print(f"F1     = {f1_value}")

Recall = 2/4 = 0.5
F1     = 0.44


### C2・撈多 vs 撈精（不用填，直接跑）

**盯著數字翻轉：** 撈精 → P=**1.0** / R=**0.5**；撈多 → P=**0.5** / R=**1.0**。

> **兩者天生拉扯、沒有兩全，只有依場景取捨。**

In [23]:
def eval_pr(retrieved, relevant):
    hit = []
    for d in retrieved:
        if d in relevant:
            hit.append(d)
    if len(retrieved) == 0:    # 💡 什麼都沒撈時分母是 0，先擋掉避免除以 0
        p = 0
    else:
        p = len(hit) / len(retrieved)
    r = len(hit) / len(relevant)
    return p, r

relevant = {"d1", "d3", "d6", "d7"}                    # 真正相關共 4 篇

# (A) 撈精：只撈最有把握的 2 篇
few = ["d1", "d3"]
# (B) 撈多：把疑似相關的全撈，8 篇
many = ["d1", "d2", "d3", "d4", "d5", "d6", "d7", "d8"]

for name, r_list in [("撈精(2篇)", few), ("撈多(8篇)", many)]:
    p, r = eval_pr(r_list, relevant)
    p2 = round(p, 2)
    r2 = round(r, 2)
    print(f"{name}: Precision={p2}  Recall={r2}")

撈精(2篇): Precision=1.0  Recall=0.5
撈多(8篇): Precision=0.5  Recall=1.0


### 📝 小作業 C

1. 系統這次只回 **top-3**（`["d1", "d2", "d3"]`，relevant 不變）——用 `eval_pr()` 算 P/R。跟 C1「撈 5 篇」比，**哪個指標動了、為什麼？**
2. **紙筆討論題（先想理由再看解）：** 某銀行做「法遵／反洗錢檢索系統」，讓法遵人員輸入條件就撈出**所有可能相關**的可疑交易紀錄與適用法規供查核。設計時**最該優先確保**哪個指標、為什麼？
   - (A) precision，因為撈上來的要乾淨
   - (B) recall，因為漏掉一筆關鍵可疑交易或一條適用法規，可能違規挨罰、代價極大
   - (C) 兩個都不重要，速度最重要
   - (D) precision，因為法遵人員沒時間看太多

> 🆕 **法遵（Compliance）** ＝確認公司做的事都符合法規；「**可疑交易**」＝像洗錢那類異常金流，金融機構依法必須找出來通報。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
p, r = eval_pr(["d1", "d2", "d3"], relevant)
print("top-3: Precision =", round(p, 2), " Recall =", round(r, 2))
```
輸出：`top-3: Precision = 0.67  Recall = 0.5`

- **Q1：動的是 precision，recall 不動。** 撈得少 → precision 的分母（我撈的）從 5 變 3 → 2/3 ≈ 0.67 升上來；但魚還是只撈到 2/4 條 → recall 停在 0.5。**（撈精 → precision 升、recall 不變或降。）**
- **Q2：答案是 (B) recall。** 金融法遵／反洗錢場景，「**漏掉**」一筆關鍵可疑交易或一條適用法規的代價（違規受罰、洗錢漏查、商譽損失）遠大於「多撈幾筆讓法遵人員過濾」→ 寧可 recall 拉高（盡量別漏）、容忍 precision 低一些。
  - 🔑 **考點：** precision／recall 沒有「誰一定重要」——**由場景的「錯誤代價」決定**：漏掉代價高 → 重 recall（法遵／反洗錢／資安）；撈錯代價高或使用者只看前幾筆 → 重 precision（一般新聞搜尋）。(D) 雖然提 precision，但理由錯置了場景優先級。
</details>

In [24]:
# 📝 小作業 C 參考解（參考解不只一種）

# Q1：只回 top-3 —— retrieved 變小、relevant 不變
p, r = eval_pr(["d1", "d2", "d3"], relevant)
p2 = round(p, 2)
r2 = round(r, 2)
print(f"top-3: Precision={p2}  Recall={r2}")
# 💡 撈得少 → precision 的分母（我撈的）變小 → 升到 2/3≈0.67；
#    但魚還是只撈到 2/4 條 → recall 停在 0.5。動的是 precision，recall 不動。

# Q2：答案是 (B) recall。
# 金融法遵／反洗錢場景，「漏掉」一筆關鍵可疑交易或一條適用法規的代價
# （違規受罰、洗錢漏查、商譽損失）遠大於「多撈幾筆讓法遵人員過濾」
# → 寧可 recall 拉高（盡量別漏）、容忍 precision 低一些。
# 🔑 考點：precision/recall 沒有「誰一定重要」——由場景的「錯誤代價」決定：
#    漏掉代價高 → 重 recall（法遵/反洗錢/資安）；撈錯代價高或只看前幾筆 → 重 precision（一般新聞搜尋）。
print("Q2 答案：(B) recall —— 法遵場景「漏掉」的代價最大，寧可多撈、別漏")

top-3: Precision=0.67  Recall=0.5
Q2 答案：(B) recall —— 法遵場景「漏掉」的代價最大，寧可多撈、別漏


---
## 🟫 練 D：讀真實中文新聞檔 + 整合搜尋

把寫死的 5 句換成 `data/news/` 裡的 **15 篇真實中文新聞**。

> ⚠️ **真實中文沒有空白**——直接餵會把整句當成一個詞，查什麼都比不到。**中文檢索第一步＝先斷詞**（用 `jieba`）。

先跑下面的 jieba 檢查格；裝不起來也別慌——小作業 D 的 ⭐ 有不用 jieba 的 Backup。

In [25]:
# ✅ 練 D 環境檢查：中文斷詞需要 jieba（練 A–C 用不到，所以放到這裡才檢查）
try:
    import jieba
    print("jieba OK，版本：", jieba.__version__)
except ModuleNotFoundError:
    print("❌ 沒裝 jieba → 終端機執行：pip install jieba")
    print("   真的裝不起來也別慌——小作業 D 的 ⭐ 有「完全不用 jieba」的 char n-gram Backup。")

jieba OK，版本： 0.42.1


/home/barai/.local/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### D1・jieba 斷詞 + 讀檔 + 重新向量化（五格）

設詞典 → 看斷詞 → 包 `cut()` → 讀 15 篇 → 重新向量化。

> **斷詞品質決定檢索品質：**「晶圓代工」被切成「晶圓／代工」就查不到 → 用 `add_word` 釘成一個詞。

**預期輸出：** `央行 決議 升息 半碼 ， 市場 解讀 偏鷹`　→　`讀到 15 篇新聞`　→　`向量矩陣形狀： (15, 320)`

> 🛟 `FileNotFoundError` → 確認 notebook 是在 `Lab1_古典IR/` 資料夾底下開的。開檔記得 `encoding="utf-8"`。

### D1-1a・先看一眼：jieba 直接斷詞（**還沒加詞典**）

中文沒有空白，得靠斷詞器切。先讓 jieba **原封不動**斷一句金融新聞看看。

**預期輸出：** 一串詞。**盯著兩個地方**——
`晶圓代工` 被切成了 `晶圓` / `代工`，`監理沙盒` 被切成了 `監理` / `沙盒`。

> ⚠️ 專有名詞被切散＝**搜尋時就比不到**了。下一格解決它。

In [26]:
# D1-1a：jieba 原味斷詞（還沒加任何詞典）
import glob, os, jieba   # ⚠️ 需先安裝：pip install jieba

sentence = "晶圓代工報價上揚，監理沙盒開放電動車保險試辦"

jieba.lcut(sentence)   # 💡 lcut ＝ 斷詞後直接回一個「詞的清單」（l ＝ list）
# 💡 看結果：「晶圓代工」被切成「晶圓 / 代工」、「監理沙盒」被切成「監理 / 沙盒」
#    ——jieba 沒看過這些金融專有名詞，只好按通用規則硬切
# 最後一行不寫 print——Jupyter 會直接顯示

Building prefix dict from the default dictionary ...


Loading model from cache /tmp/jieba.cache


Loading model cost 0.305 seconds.


Prefix dict has been built successfully.


['晶圓', '代工', '報價', '上揚', '，', '監理', '沙盒', '開放', '電動車', '保險', '試辦']

### D1-1b・加上自訂詞典，**同一句再斷一次**

把金融專有名詞先「釘」成一個詞，再斷同一句。

**預期輸出：** `晶圓代工` 和 `監理沙盒` 這次**各自是一個完整的詞**了。

> 這就是自訂詞典的價值——**兩格對照著看，差別一目了然**。

In [27]:
# D1-1b：把專有名詞加進詞典，再斷同一句
for w in ["晶圓代工", "毛利率", "半導體", "升息", "央行", "監理沙盒", "電動車", "再生能源", "每股盈餘"]:
    jieba.add_word(w)   # 💡 add_word＝把這個詞「釘」進詞典，斷詞時不會再被拆開
# 詞多時可寫成一個檔，用 jieba.load_userdict("fin_dict.txt") 一次載入

jieba.lcut(sentence)   # 💡 同一句、同一個方法——差別只在「詞典裡多了那幾個詞」
# 💡 對照上一格：「晶圓代工」「監理沙盒」現在各自完整了
#    → 專有名詞是搜尋的關鍵字，切散了就永遠比不到
# 最後一行不寫 print——Jupyter 會直接顯示

['晶圓代工', '報價', '上揚', '，', '監理沙盒', '開放', '電動車', '保險', '試辦']

In [28]:
# D1-1c：包成 cut()——斷詞後用空白接回成一個字串
def cut(text):
    return " ".join(jieba.lcut(text.strip()))  # 💡 斷詞後用空白接起來，才餵得進 token_pattern=\S+
#        💡 D1-1b 看到的那串詞，用 " ".join(...) 接成一個字串——這就是 TfidfVectorizer 要的形式

print(cut("央行決議升息半碼，市場解讀偏鷹"))   # 看到詞與詞之間被空白隔開就對了

央行 決議 升息 半碼 ， 市場 解讀 偏鷹


In [29]:
# D1-2a：glob 讀 data/news/*.txt（讀進來先斷詞）
paths = sorted(glob.glob("data/news/*.txt"))
docs = []
names = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        docs.append(cut(f.read()))             # ← 關鍵：讀進來「先斷詞」再放進語料
        # （斷詞在「進語料前」做一次就好——之後向量化、查詢都吃「空白分隔」的形式）
        names.append(os.path.basename(p))

n_docs = len(docs)
first3 = names[:3]
print(f"讀到 {n_docs} 篇新聞：{first3} ...")

讀到 15 篇新聞：['001_升息.txt', '002_台股大盤.txt', '003_半導體.txt'] ...


### D1-2b・⚠️ 換語料了，該用哪一個？（**故意先錯一次**）

語料從「5 句短句」換成「15 篇新聞」了。這裡**故意先用錯的那個**，看看會發生什麼事。

**預期輸出：** 形狀 `(15, 23)`。

> 🔍 **盯住第二個數字：23。** 那是**舊那 5 句短句**的詞彙表大小——新語料有幾百個詞，卻硬被塞進舊的 23 個格子裡。
> **錯了會當場現形**，不必等到查不到東西才發現。

In [30]:
# D1-2b：故意用「只轉、不學」的那個——看它壞在哪
doc_vectors = vectorizer.transform(docs)   # 💡 transform ＝照「舊的」詞彙表轉換，不會學新詞

print("向量矩陣形狀：", doc_vectors.shape)
print("⚠️ 第二個數字是 23 嗎？那是舊語料（5 句短句）的詞彙表大小——新語料的詞根本沒被學進去")
# 💡 這就是選錯的下場：15 篇新聞被硬塞進舊的 23 個格子，大部分內容直接消失

向量矩陣形狀： (15, 23)
⚠️ 第二個數字是 23 嗎？那是舊語料（5 句短句）的詞彙表大小——新語料的詞根本沒被學進去


### D1-2c・改用正確的那個

**預期輸出：** 形狀 `(15, 320)`。

> 🔑 **23 → 320**：詞彙表真的重學過了。
> **規則再說一次：換語料 ＝ 一定要重新「學」一次。**

In [31]:
# D1-2c：換成正確的——重新學一份詞彙表
doc_vectors = vectorizer.fit_transform(docs)   # 💡 換語料一定要重跑 fit_transform（重新學詞彙表 + IDF）

print("向量矩陣形狀：", doc_vectors.shape)
print("✅ 第二個數字變成三位數了嗎？代表 15 篇新聞的詞彙表真的重學過了")
# 💡 23 → 320：對照上一格，一眼就看得出「有沒有學」的差別

向量矩陣形狀： (15, 320)
✅ 第二個數字變成三位數了嗎？代表 15 篇新聞的詞彙表真的重學過了


### D2・整合成 `search_files()`：一條龍搜尋

把所有零件接成一個能搜「整個資料夾」的小引擎。

> ⚠️ **查詢也要先斷詞**——文件走了 `cut()`、查詢沒走，就對不上同一份詞彙表，查什麼都比不到。

**預期輸出：** 查「升息」→ `001_升息.txt` 第 1；查「台股 半導體」→ `002`／`003` 前兩名。

In [32]:
def search_files(query, top_k=3):
    qv = vectorizer.transform([cut(query)])    # ⚠️ 查詢也要先斷詞！（和文件用同一套 cut）
    # 💡 文件、查詢必須走同一套 cut()，切出來的詞才對得上同一份詞彙表
    scores = cosine_similarity(qv, doc_vectors)[0]
    score_list = []
    for doc_id, score in enumerate(scores):
        score_list.append((score, doc_id))
    ranking = sorted(score_list, reverse=True)

    print("")
    print(f"查詢：「{query}」")
    shown = 0
    for score, doc_id in ranking:
        if score > 0 and shown < top_k:
            shown = shown + 1
            s = round(score, 3)
            name = names[doc_id]
            snippet = docs[doc_id][:24]
            print(f"  第{shown}名 分數={s} 【{name}】{snippet}...")

search_files("升息")
search_files("台股 半導體")


查詢：「升息」
  第1名 分數=0.233 【001_升息.txt】【 虛構 示意 ・ 非真實 行情 】 央行 理事...
  第2名 分數=0.144 【007_金融銀行.txt】【 虛構 示意 】 升息 環境 下 銀行 利差 ...
  第3名 分數=0.144 【013_通膨物價.txt】【 虛構 示意 】 最新 物價 數據 顯示 通膨...

查詢：「台股 半導體」
  第1名 分數=0.251 【002_台股大盤.txt】【 虛構 示意 】 台股 今日 開高 走高 ， ...
  第2名 分數=0.201 【003_半導體.txt】【 虛構 示意 】 晶圓代工 報價 傳出 調漲 ...


### 📝 小作業 D

1. 用 `search_files()` 查「**電動車**」——第 1 名是不是 `005_電動車.txt`？
2. **觀察題：** 再查「**綠能 政策**」——第 1 名**很可能不是** `015_綠能.txt`。跑出來後，打開 `data/news/015_綠能.txt` 看原文，想想為什麼？（提示：015 篇的原文用的是哪些詞？查詢裡真正「比中」的是哪個詞？）
3. **⭐ 進階（選做）：不用 jieba 的 Backup。** 用 `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))`（「字的 2~3 連字組合」當特徵，**免斷詞、零安裝**）重建一套索引查「升息」，跟 jieba 版比排序。想一想：char_wb 吃的是**沒斷詞的原文**，那你手上的 `docs` 還能用嗎？（它已經被 `cut()` 過了）

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
search_files("電動車")      # Q1：005 穩排第 1
search_files("綠能 政策")   # Q2：第 1 名不是 015_綠能.txt！

# ⭐ 進階：char n-gram——注意要「重讀一次原文」，因為 docs 已經被 cut() 斷過詞了
raw_docs = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        raw_docs.append(f.read())

char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
char_vectors = char_vec.fit_transform(raw_docs)
qv = char_vec.transform(["升息"])
scores = cosine_similarity(qv, char_vectors)[0]
score_list = []
for doc_id, score in enumerate(scores):
    score_list.append((score, doc_id))
ranking = sorted(score_list, reverse=True)
for score, doc_id in ranking[:3]:
    print(names[doc_id], round(score, 3))
```

- **Q1：** 「電動車」在 005 篇反覆出現 → 第 1 名，排序合理。
- **Q2（重點）：** 「綠能 政策」的第 1 名**不是** `015_綠能.txt`。打開 015 原文會發現：它寫的是「**再生能源**」「**綠電**」「**太陽能**」——**字面上幾乎沒有「綠能」這個詞**！查詢裡真正比中的只剩「政策」，而 005（補助政策）、004（減產政策）也都含「政策」→ 015 反而排不到第 1。
  - 🔑 **這就是 TF-IDF「關鍵字比對」的天花板：** 「綠能」和「再生能源」意思一樣、字面不同就比不到。**下午的 embedding（語意檢索）正是為了解決這件事——這題請記住，下午會回收。**
- **⭐ 進階：** char n-gram 版查「升息」第 1 名仍是 001，但雜訊比 jieba 版多（比較粗）——它的價值是**完全不需要斷詞器**，教室機裝不了 jieba 時的救生圈。
</details>

In [33]:
# 📝 小作業 D 參考解（參考解不只一種）

# Q1：「電動車」一詞就在 005 篇裡反覆出現 → 005 穩排第 1，排序合理
search_files("電動車")

# Q2（觀察題）：「綠能 政策」的第 1 名不是 015_綠能.txt！
search_files("綠能 政策")
# 💡 為什麼？打開 015 原文：它寫的是「再生能源」「綠電」「太陽能」——
#    字面上幾乎沒有「綠能」這個詞！查詢裡真正比中的只剩「政策」，
#    而 005（補助政策）、004（減產政策）也都含「政策」→ 015 反而排不到第 1。
# 🔑 這就是 TF-IDF「關鍵字比對」的天花板：「綠能」和「再生能源」意思一樣、字面不同就比不到。
#    下午的 embedding（語意檢索）正是為了解決這件事——這題請記住，下午會回收。


查詢：「電動車」
  第1名 分數=0.21 【005_電動車.txt】【 虛構 示意 】 電動車 買氣 回溫 ， 電池...

查詢：「綠能 政策」
  第1名 分數=0.163 【005_電動車.txt】【 虛構 示意 】 電動車 買氣 回溫 ， 電池...
  第2名 分數=0.155 【004_原油能源.txt】【 虛構 示意 】 國際 原油 庫存 下滑 ， ...
  第3名 分數=0.148 【015_綠能.txt】【 虛構 示意 】 再生能源 建置 加速 ， 太...


In [34]:
# ⭐ 小作業 D 進階（選做）參考解：不用 jieba 的 Backup——char n-gram

# char_wb 吃的是「沒斷詞的原文」——docs 已被 cut() 斷過詞了，所以要重讀一次原始檔案
raw_docs = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        raw_docs.append(f.read())

char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))   # 💡 「字的 2~3 連字組合」當特徵，免斷詞、零安裝
char_vectors = char_vec.fit_transform(raw_docs)
qv = char_vec.transform(["升息"])
scores = cosine_similarity(qv, char_vectors)[0]

score_list = []
for doc_id, score in enumerate(scores):      # 💡 enumerate：一次拿到「編號 + 分數」
    score_list.append((score, doc_id))       # 💡 先組 (分數, 編號) tuple 清單，等下好排序
ranking = sorted(score_list, reverse=True)   # 💡 分數大的排前面
for score, doc_id in ranking[:3]:
    print(names[doc_id], round(score, 3))

# 結論：char n-gram 查「升息」第 1 名仍是 001，但雜訊比 jieba 版多（比較粗）。
#       它的價值＝完全不需要斷詞器，是教室機裝不了 jieba 時的救生圈。

001_升息.txt 0.117
013_通膨物價.txt 0.069
008_房市營建.txt 0.068


---
## 🚀 附錄（選做）：BM25 排序對比

> 需先 `pip install rank_bm25`；卡在前面的直接跳過，不影響核心。

業界排序實際多用 TF-IDF 的升級版 **BM25**（`Elasticsearch` 預設就是它）。

**看什麼：** BM25 分數**不是 0~1**、跟 cosine 是兩套尺度，但**排序方向一樣**——查「升息」仍是文件 0 第 1、文件 4 第 2。

In [35]:
# BM25 附錄用回練 A/B 的 5 篇短句語料（練 D 已把 docs 換成 15 篇新聞，這裡先換回來，
# 才能跟 B3 的 TF-IDF 排序結果（文件0 → 文件4）直接對照）
docs = [
    "升息 抑制 通膨 央行 升息 一碼",          # 文件0：升息
    "台股 開高 半導體 領漲 台股 收紅",        # 文件1：台股/半導體
    "原油 庫存 下滑 油價 走高",               # 文件2：原油
    "半導體 毛利率 提升 晶圓代工 報價 上揚",   # 文件3：半導體財報
    "升息 衝擊 台股 資金 外流",               # 文件4：升息+台股
]
print("已把 docs 換回 5 篇短句語料")

已把 docs 換回 5 篇短句語料


In [36]:
# pip install rank_bm25
from rank_bm25 import BM25Okapi

tokenized_docs = []
for d in docs:
    tokenized_docs.append(d.split())   # BM25 吃「已斷詞的 list」——每篇是一串詞
bm25 = BM25Okapi(tokenized_docs)

q = "升息".split()
bm25_scores = bm25.get_scores(q)
# 💡 BM25 分數不是 0~1（跟 cosine 是兩套尺度）——只看排序方向、不對數字

score_list = []
for doc_id, score in enumerate(bm25_scores):
    score_list.append((score, doc_id))
ranking = sorted(score_list, reverse=True)

print("BM25 排序（查「升息」）：")
rank = 1
for score, doc_id in ranking:
    if score > 0:
        s = round(score, 3)
        text = docs[doc_id]
        print(f"第{rank}名  BM25分數={s}  文件{doc_id}: {text}")
        rank = rank + 1

BM25 排序（查「升息」）：
第1名  BM25分數=0.47  文件0: 升息 抑制 通膨 央行 升息 一碼
第2名  BM25分數=0.354  文件4: 升息 衝擊 台股 資金 外流


---
## 🗺 總結：你今天親手做了一個古典搜尋引擎

| 練 | 你做了什麼 | 在搜尋引擎裡的角色 |
|---|---|---|
| A | `dict` 手刻倒排索引 + 布林 AND/OR | **秒查**哪些文件含這個詞 |
| B | `TfidfVectorizer` + cosine + `search()` | **排序**——回答「多相關」 |
| C | 手算 P/R + `eval_pr()` | **評估**——準不準、有沒有漏 |
| D | jieba 斷詞 + 讀 15 篇真實中文新聞 | **真實語料**——斷詞品質決定檢索品質 |

**🔗 接下午：** 你做的是「把文件變向量、用 cosine 比相似」——但 TF-IDF 比的是**字面**：搜「升息」找不到寫「調升利率」的文件。下午把 `TfidfVectorizer` 換成 **embedding 模型**，同一套骨架就升級成**語意檢索**——**Lab 3 的 RAG，「檢索」那一半用的正是這個流程。**